# IEEE-CIS Preprocessing

Cleans, imputes, scales and splits the IEEE-CIS fraud dataset, then balances the training pool with SMOTENC.

Produces `preprocessed.pkl`, consumed by `IEEE-CIS-SET1.ipynb`, `IEEE-CIS-SET2.ipynb` and `preprocess_patterns.ipynb`.

In [ ]:
# load raw transaction + identity CSVs
import pandas as pd
import numpy as np

print("Loading dataset partitions with optimized datatypes...")

dtype_dict = {
    'TransactionID': np.int32,
    'isFraud': np.int8,
    'TransactionDT': np.int32,
    'TransactionAmt': np.float32
}

train_transaction = pd.read_csv(
    "train_transaction_IEEE.csv", 
    engine="c", 
    low_memory=True,
    memory_map=True,
    dtype=dtype_dict
)

train_identity = pd.read_csv(
    "train_identity_IEEE.csv", 
    engine="c", 
    low_memory=True,
    memory_map=True,
    dtype={'TransactionID': np.int32}
)

print("train_transaction shape:", train_transaction.shape)
print("train_identity shape:", train_identity.shape)

In [ ]:
# left-join transaction and identity on TransactionID
df = train_transaction.merge(train_identity, on="TransactionID", how="left")

print("Merged shape:", df.shape)
df.head()

In [ ]:
# check merged shape
print("Total columns:", df.shape[1])
print("Total rows:", df.shape[0])

In [ ]:
# summarise missing values per column
null_counts = df.isnull().sum().sort_values(ascending=False)
null_percent = (null_counts / len(df) * 100).round(2)

null_summary = pd.DataFrame({
    "null_count": null_counts,
    "null_percent": null_percent
})

null_summary = null_summary[null_summary.null_count > 0]

print("Total columns with missing values:", len(null_summary), "out of", df.shape[1])
print("Columns >90% missing:", (null_summary.null_percent > 90).sum())
print("Columns 50-90% missing:", ((null_summary.null_percent > 50) & (null_summary.null_percent <= 90)).sum())
print("Columns 10-50% missing:", ((null_summary.null_percent > 10) & (null_summary.null_percent <= 50)).sum())
print("Columns <10% missing:", (null_summary.null_percent <= 10).sum())

In [ ]:
# drop columns with >90% missing values
high_missing = null_summary[null_summary.null_percent > 90].index.tolist()

print("Dropping columns (>90% missing):")
print(high_missing)

df_clean = df.drop(columns=high_missing)

print("Shape before:", df.shape)
print("Shape after:", df_clean.shape)

In [ ]:
# check for duplicate rows / TransactionIDs
dup_rows = df.duplicated().sum()
dup_ids = df["TransactionID"].duplicated().sum()

print(f"Fully duplicated rows: {dup_rows}")
print(f"Duplicated TransactionIDs: {dup_ids}")

In [ ]:
# class balance of isFraud
print(df["isFraud"].value_counts())
print((df["isFraud"].value_counts(normalize=True) * 100).round(2))

In [ ]:
# print dataset summary
print("IEEE-CIS DATASET SUMMARY")
print(f"  Total transactions   : {len(df_clean):,}")
print(f"  Legitimate (0)       : {(df_clean['isFraud']==0).sum():,}")
print(f"  Fraud (1)            : {(df_clean['isFraud']==1).sum():,}")
print(f"  Fraud rate           : {df_clean['isFraud'].mean()*100:.4f}%")
print(f"  Number of features   : {df_clean.shape[1]-1}")
print(f"  Missing values       : {df_clean.isnull().sum().sum()}")
print(f"  Columns dropped      : 12 (>90% missing)")
print("Preprocessing complete. Ready for SMOTE.")

In [ ]:
# plot legit vs fraud class distribution
import matplotlib.pyplot as plt

legit_count = (df_clean['isFraud'] == 0).sum()
fraud_count = (df_clean['isFraud'] == 1).sum()
total = len(df_clean)

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(
    ['Legitimate', 'Fraud'],
    [legit_count, fraud_count],
    color=['steelblue', 'crimson'],
    width=0.5,
    edgecolor='white'
)

for bar, value in zip(bars, [legit_count, fraud_count]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1500,
        f'{value:,}\n({value/total*100:.3f}%)',
        ha='center', va='bottom',
        fontsize=12, fontweight='bold'
    )

ax.set_title('IEEE-CIS Transaction Class Distribution',
             fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Number of Transactions', fontsize=12)
ax.set_ylim(0, legit_count * 1.15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# save cleaned, unscaled snapshot (needed later for behavioural features)
df_clean.to_parquet("merged_data.parquet", index=False)
print("Saved unscaled cleaned data:", df_clean.shape)

In [ ]:
# impute missing values (mean for numeric, mode for categorical)
from sklearn.impute import SimpleImputer

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()

numeric_cols = [col for col in numeric_cols if col != 'isFraud']

print(f"Numeric columns to impute: {len(numeric_cols)}")
print(f"Categorical columns to impute: {len(categorical_cols)}")

mean_imputer = SimpleImputer(strategy='mean')
df_clean[numeric_cols] = mean_imputer.fit_transform(df_clean[numeric_cols])

mode_imputer = SimpleImputer(strategy='most_frequent')
df_clean[categorical_cols] = mode_imputer.fit_transform(df_clean[categorical_cols])

remaining_nulls = df_clean.isnull().sum().sum()
print(f"\nMissing values remaining after imputation: {remaining_nulls}")
print(f"Shape unchanged: {df_clean.shape}")

In [ ]:
# check which columns will be scaled vs left alone
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols_to_scale = [col for col in numeric_cols if col != 'isFraud']

categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()

print(f"Total columns in df_clean: {df_clean.shape[1]}")
print(f"\nNumeric columns (to be scaled): {len(numeric_cols_to_scale)}")
print(f"Categorical columns (NOT scaled): {len(categorical_cols)}")
print(f"Target column isFraud (NOT scaled): 1")
print(f"\nCheck: {len(numeric_cols_to_scale)} + {len(categorical_cols)} + 1 = {len(numeric_cols_to_scale) + len(categorical_cols) + 1}")

print(f"\nCategorical columns won't be scaled):")
print(categorical_cols)

In [ ]:
# standard-scale numeric columns (excludes isFraud, drops TransactionID)
from sklearn.preprocessing import StandardScaler

df_clean = df_clean.drop(columns=['TransactionID'], errors='ignore')

numeric_cols_to_scale = [col for col in df_clean.select_dtypes(
    include=[np.number]).columns if col != 'isFraud']

print(f"Scaling {len(numeric_cols_to_scale)} numeric columns...")

scaler = StandardScaler()
df_clean[numeric_cols_to_scale] = scaler.fit_transform(
    df_clean[numeric_cols_to_scale])

check_cols = ['TransactionAmt', 'C1', 'V1']
print("\nVerification (mean≈0, std≈1):")
print(df_clean[check_cols].describe().loc[['mean','std']].round(4))
print(f"\nShape after scaling: {df_clean.shape}")
print("Scaling complete - excluding TransactionID column")

In [ ]:
# carve out test set, build pre-SMOTE train pool
import pandas as pd
import numpy as np

TEST_SIZE = 30000

df_test = df_clean.sample(n=TEST_SIZE, random_state=42)
df_remaining = df_clean.drop(df_test.index)

print("=== Test Set ===")
print("Total:", len(df_test))
print("Fraud:", df_test['isFraud'].sum(),
      f"({df_test['isFraud'].mean()*100:.2f}%)")

X_test_raw = df_test.drop(columns=['isFraud'])
y_test = df_test['isFraud'].values

TARGET_FRAUD = 28000
TARGET_LEGIT = 42000

fraud_pool = df_remaining[df_remaining['isFraud'] == 1]
legit_pool = df_remaining[df_remaining['isFraud'] == 0]

fraud_sample = fraud_pool.copy()
legit_sample = legit_pool.sample(n=TARGET_LEGIT, random_state=42)

df_train = pd.concat([fraud_sample, legit_sample])

df_train = df_train.sample(frac=1, random_state=42)

print("\n=== Pre-SMOTE Training Pool ===")
print("Real fraud:", len(fraud_sample))
print("Real legit:", len(legit_sample))
print("Total:", len(df_train))
print("Synthetic fraud needed:", TARGET_FRAUD - len(fraud_sample))

In [ ]:
# oversample minority class with SMOTENC
from imblearn.over_sampling import SMOTENC
import numpy as np

X_train = df_train.drop(columns=['isFraud'])
y_train = df_train['isFraud']

categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
categorical_indices = [X_train.columns.get_loc(c) for c in categorical_cols]

print("Before SMOTENC:")
print("  Legit:", (y_train==0).sum(), "| Fraud:", (y_train==1).sum())

smote_nc = SMOTENC(
    categorical_features=categorical_indices,
    sampling_strategy=TARGET_FRAUD / TARGET_LEGIT,
    random_state=42,
    k_neighbors=5
)

X_train_smote, y_train_smote = smote_nc.fit_resample(X_train, y_train)

print("\nAfter SMOTENC:")
print("  Legit:", (y_train_smote==0).sum(), "| Fraud:", (y_train_smote==1).sum())
print("  Total:", len(y_train_smote))
print("  Fraud %:", round(y_train_smote.mean()*100, 1))
print("  Synthetic rows generated:", len(X_train_smote) - len(X_train))

In [ ]:
# save preprocessed data + split metadata to disk
import pickle

preprocessed_data = {
    "df_clean": df_clean,
    "df_train": df_train,
    "df_test": df_test,
    "X_train_smote": X_train_smote,
    "y_train_smote": y_train_smote,
    "TARGET_FRAUD": TARGET_FRAUD,
    "TARGET_LEGIT": TARGET_LEGIT,
}

with open("preprocessed.pkl", "wb") as f:
    pickle.dump(preprocessed_data, f)

In [ ]:
# recover TransactionAmt mean/std used by the scaler
amt_col_idx = list(numeric_cols_to_scale).index('TransactionAmt')
amt_mean = scaler.mean_[amt_col_idx]
amt_std = scaler.scale_[amt_col_idx]
print(f"TransactionAmt mean: {amt_mean}")
print(f"TransactionAmt std:  {amt_std}")

In [ ]:
# find a high-risk demo transaction for the dashboard
import numpy as np
import pickle
import tensorflow as tf

model = tf.keras.models.load_model("dashboard_model.keras")
with open("dashboard_features.pkl", "rb") as f:
    feature_names = pickle.load(f)
demo_X = np.load("dashboard_demo_X.npy")
demo_y = np.load("dashboard_demo_y.npy")

probs = model.predict(demo_X, verbose=0).flatten()
best_idx = np.argmax(probs)

print(f"Highest-risk transaction found: index {best_idx}")
print(f"Model risk score: {probs[best_idx]*100:.1f}%")
print(f"Actual label: {'Fraud' if demo_y[best_idx] == 1 else 'Legit'}")
print()

AMT_MEAN = 135.02
AMT_STD = 239.16

amt_idx = feature_names.index("TransactionAmt")
scaled_amt = demo_X[best_idx, amt_idx]
real_amt = scaled_amt * AMT_STD + AMT_MEAN
print(f"TransactionAmt: ${real_amt:.2f}")

for prefix, label in [("card4_", "card4"), ("card6_", "card6"), ("R_emaildomain_", "R_emaildomain")]:
    for f in feature_names:
        if f.startswith(prefix):
            idx = feature_names.index(f)
            if demo_X[best_idx, idx] == 1:
                print(f"{label}: {f[len(prefix):]}")